# Aim 2 DL arm: Olsen 2020 BiGRU training (Colab)

Trains `OlsenBiGRU` (`thesis_pipeline/dl_models.py`) on packed stage-B arrays with
`scripts/train_aim2_dl.py`. Design: vault pre-registration note plus the build design
(sections 1, 2.3-2.4, 4, 5).

**Configs:** M = [RR, EDR] (faithful Olsen), P4 = [RR, EDR, THOR, ABDO]. F6 is **outside the
pre-registration** (Aim 1 airflow picker defect): no F6 model is trained here and
`--allow-deferred` is never used. **Seeds:** 42, 43, 44 (model seed; the split seed stays 42).

**Code pin:** every session checks out the git tag `TRAIN_REF` (`aim2-dl-train-v1`, the training
commit C_train) detached, and asserts HEAD equals it. All production runs of M and P4 use that one
commit; there is no `git pull` to a moving HEAD. The tag must contain the frozen pre-registration
record `prereg/aim2_dl_olsen_v1_freeze.json`: `train_aim2_dl.py` refuses to train without it
(`--allow-unfrozen` is for pilots only and is never used here).

**Test protection (hard rules):**
* Drive holds **train and val only** (`MyDrive/Thesis/data/dl_cache_v1/model_v1/{train,val}`).
  This notebook asserts there is no `test` folder and never copies one.
* No test inference here. Test predictions are made once, after the code freeze, on the Mac
  (`scripts/predict_aim2_dl.py --split test --freeze-tag ...`), and metrics only by
  `scripts/evaluate_aim2_dl.py`.
* Pilot / smoke numbers are for debugging and are never reported.

**Order:** cells 1-5 every session; cell 6 (gate) once, which locks the recipe for all
configs and seeds; cell 7 trains one seed with auto-resume (re-run after a disconnect);
cell 8 runs three seeds concurrently; cell 9 summarises (warning on any run not trained at
C_train), locks the aggregator from M seed 42 (with its best-checkpoint sha256) and re-finalises
(no retraining) any run whose aggregator differs from the lock. Cell 10 lists the Mac steps.

Outputs are written to VM-local disk (`/content/out`, `/content/ckpt`) and mirrored to Drive
(`RESULTS/<config>/seed<k>`). A Drive I/O error is logged and retried on the next save; it
does not stop training.

In [ ]:
# 1. Mount Drive and fix paths
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, json, time, hashlib, shutil, subprocess, pathlib

DRIVE_ROOT = '/content/drive/MyDrive/Thesis'
DATA_DRIVE = f'{DRIVE_ROOT}/data/dl_cache_v1/model_v1'      # train/ and val/ ONLY
RESULTS = f'{DRIVE_ROOT}/results/aim2_dl_olsen_v1'
LOGS = f'{DRIVE_ROOT}/logs/aim2_dl_olsen_v1'
LOCAL = '/content/dl_cache/model_v1'
CKPT_LOCAL = '/content/ckpt'
OUT_LOCAL = '/content/out'                           # local outputs, mirrored to RESULTS
for d in (RESULTS, LOGS):
    os.makedirs(d, exist_ok=True)
assert os.path.isdir(f'{DATA_DRIVE}/train') and os.path.isdir(f'{DATA_DRIVE}/val'), 'upload train/ and val/ first'
assert not os.path.exists(f'{DATA_DRIVE}/test'), 'TEST DATA MUST NEVER BE ON DRIVE: remove it before continuing'
print('Drive OK:', DATA_DRIVE, '->', RESULTS)

In [ ]:
# 2. Clone the repo and check out the pinned training commit C_train (tag TRAIN_REF). Never pull a moving HEAD.
from google.colab import userdata
PAT = userdata.get('GITHUB_PAT_THESIS')
REPO = 'jameswudo1019hack/thesis-bme-pipeline'
TRAIN_REF = 'aim2-dl-train-v1'   # git tag of C_train: ONE commit for every production run of M and P4

def run_git(cmd, what):
    """Run git; on failure raise with git's own stderr (PAT redacted) instead of a bare exit status."""
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        err = (r.stderr or r.stdout).strip()
        raise RuntimeError(f'git {what} failed (exit {r.returncode}): {err.replace(PAT, "***") if PAT else err}')
    return r.stdout.strip()

def git(*args):
    return run_git(['git', '-C', '/content/Code', *args], ' '.join(args))

if not pathlib.Path('/content/Code').exists():  # full history (no --depth), so every tag is reachable
    run_git(['git', 'clone', f'https://{PAT}@github.com/{REPO}.git', '/content/Code'], f'clone {REPO}')
git('fetch', '--tags', 'origin')
git('checkout', '--detach', TRAIN_REF)
COMMIT = git('rev-parse', 'HEAD')
assert COMMIT == git('rev-list', '-n', '1', TRAIN_REF), f'HEAD {COMMIT} is not {TRAIN_REF}'
assert not git('status', '--porcelain', '--untracked-files=no'), 'tracked files modified in /content/Code'
for f in ('thesis_pipeline/dl_train.py', 'thesis_pipeline/dl_models.py', 'scripts/train_aim2_dl.py',
          'scripts/bench_bigru.py', 'splits/aim2_seed42.json', 'prereg/aim2_dl_olsen_v1_freeze.json'):
    assert pathlib.Path('/content/Code', f).exists(), f'{f} missing at {TRAIN_REF}'
FREEZE = json.load(open('/content/Code/prereg/aim2_dl_olsen_v1_freeze.json'))
print('HEAD', COMMIT, '=', TRAIN_REF, '| frozen pre-registration body', FREEZE['body_sha256'][:12],
      '| vault commit', FREEZE.get('vault_commit'))

In [ ]:
# 3. Dependencies (torch is preinstalled on Colab)
%cd /content/Code
!pip install -q neurokit2 pyarrow click scikit-learn lxml mne 2>&1 | tail -2
!pip install -q -e . 2>&1 | tail -2
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 4. Run parameters
CONFIG = 'M'                 # 'M' or 'P4' (F6 deferred)
SEEDS = [42, 43, 44]
CHANNELS = {'M': ['RR', 'EDR'], 'P4': ['RR', 'EDR', 'THOR', 'ABDO']}[CONFIG]
RECIPE_LOCK = f'{RESULTS}/RECIPE_LOCK.json'          # written once by the gate (cell 6)
AGG_LOCK = f'{RESULTS}/AGGREGATOR_LOCK.json'         # written once from M seed 42 (cell 9)
RECIPE = json.load(open(RECIPE_LOCK))['recipe'] if os.path.exists(RECIPE_LOCK) else None
print('config', CONFIG, CHANNELS, 'recipe', RECIPE)

In [ ]:
# 5. Copy the config's train/val shards to local disk and verify sha256 against MANIFEST.json
def sha256(p, buf=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(buf), b''):
            h.update(b)
    return h.hexdigest()

split_sha = sha256('/content/Code/splits/aim2_seed42.json')
t0 = time.time()
for split in ('train', 'val'):
    src, dst = f'{DATA_DRIVE}/{split}', f'{LOCAL}/{split}'
    os.makedirs(dst, exist_ok=True)
    man = json.load(open(f'{src}/MANIFEST.json'))
    assert man['split'] == split, f'{src} manifest says {man["split"]!r}'
    assert man['split_sha256'] == split_sha, 'cache packed from a different split file'
    names = [n for n in man['sha256'] if not n.startswith('signals_') or any(f'_{c}_' in n for c in CHANNELS)]
    for n in names:
        s, d = f'{src}/{n}', f'{dst}/{n}'
        if not (os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s) and sha256(d) == man['sha256'][n]):
            shutil.copyfile(s, d)
            assert sha256(d) == man['sha256'][n], f'sha256 mismatch after copy: {n}'
    shutil.copyfile(f'{src}/MANIFEST.json', f'{dst}/MANIFEST.json')
    print(split, len(names), 'files verified;', man['n_subjects'], 'subjects')
assert not os.path.exists(f'{LOCAL}/test')
print(f'copy + verify {time.time() - t0:.0f} s')

In [ ]:
# 6. Benchmark gate (ONCE, before any production run). Locks the recipe for all configs and seeds.
#    R-A: batch-128 single >= 800 win/s or 3 processes >= 1500 total; R-B: 400-800; R-C: < 400 or T4.
import sys
sys.path.insert(0, '/content/Code/scripts')
from bench_bigru import gate_recipe
if os.path.exists(RECIPE_LOCK):
    print('recipe already locked:', json.load(open(RECIPE_LOCK)))
else:
    cin = 2  # pre-registered gate: c_in = 2 (M), AMP, whatever CONFIG is selected in cell 4
    !nvidia-smi --query-gpu=name,memory.total --format=csv
    !python scripts/bench_bigru.py --device cuda --amp --cin {cin} --batch 128 --batch 256 --batch 512 --json-out /content/gate_single.json
    !python scripts/bench_bigru.py --device cuda --amp --cin {cin} --batch 128 --procs 3 --json-out /content/gate_multi.json
    !python scripts/bench_bigru.py --device cuda --amp --cin {cin} --batch 128 --cache-dir {LOCAL}/train --json-out /content/gate_mmap.json
    single = json.load(open('/content/gate_single.json'))
    multi = json.load(open('/content/gate_multi.json'))[0]
    for f in ('/content/gate_single.json', '/content/gate_multi.json', '/content/gate_mmap.json'):
        assert os.path.exists(f), f'{f} missing: that benchmark failed (see its output above); nothing locked'
    mmap = json.load(open('/content/gate_mmap.json'))[0]
    s128 = next(r['train_windows_per_sec'] for r in single if r['batch'] == 128)
    gpu = single[0]['gpu'] or ''
    RECIPE = gate_recipe(s128, multi['total_train_windows_per_sec'], t4='T4' in gpu)
    units = input('Compute-unit balance now (from the Colab resources panel): ')
    lock = {'recipe': RECIPE, 'gpu': gpu, 'single_128': s128, 'multi_total': multi['total_train_windows_per_sec'],
            'mmap_gather_128': mmap['train_windows_per_sec'], 'single': single, 'c_in': cin, 'amp': True,
            'commit': COMMIT,
            'train_ref': TRAIN_REF,
            'compute_units_at_gate': units, 'locked': time.strftime('%Y-%m-%dT%H:%M:%S')}
    json.dump(lock, open(RECIPE_LOCK, 'w'), indent=2)
    print('LOCKED recipe', RECIPE, lock)

In [ ]:
# 7. Train ONE seed with auto-resume (checkpoints and outputs mirrored to Drive every pass). Re-run after a disconnect.
assert RECIPE, 'run the gate (cell 6) first'
SEED = 42
run = f'{RESULTS}/{CONFIG}/seed{SEED}'
os.makedirs(run, exist_ok=True)
if CONFIG == 'M' and SEED == 42:
    agg = 'auto'                                   # the pre-declared rule runs on M seed-42 validation
elif os.path.exists(AGG_LOCK):
    agg = json.load(open(AGG_LOCK))['aggregator']  # frozen choice for every other run
else:
    agg = 'auto'
    print('NOTE: aggregator not locked yet; cell 9 re-finalises this run with the locked aggregator (no retraining)')
cmd = (f'python scripts/train_aim2_dl.py --config {CONFIG} --model-seed {SEED} --recipe {RECIPE} '
       f'--train-dir {LOCAL}/train --val-dir {LOCAL}/val --ckpt-dir {CKPT_LOCAL}/{CONFIG}/seed{SEED} '
       f'--mirror-dir {run} --out-dir {OUT_LOCAL}/{CONFIG}/seed{SEED} --aggregator {agg}')
!{cmd} 2>&1 | tee -a {LOGS}/train_{CONFIG}_seed{SEED}.log

In [ ]:
# 8. OPTIONAL: three seeds concurrently on one GPU (only if the gate showed >= 1.5x aggregate throughput).
#    Each process resumes from its own Drive mirror; logs go to Drive. Re-run to resume after a disconnect.
#    A process that exits with an error is restarted (it resumes) at most MAX_RESTARTS times.
assert RECIPE
MAX_RESTARTS = 2

def launch(seed):
    run = f'{RESULTS}/{CONFIG}/seed{seed}'
    os.makedirs(run, exist_ok=True)
    agg = 'auto' if (CONFIG == 'M' and seed == 42) else (
        json.load(open(AGG_LOCK))['aggregator'] if os.path.exists(AGG_LOCK) else 'auto')
    if agg == 'auto' and not (CONFIG == 'M' and seed == 42):
        print(f'NOTE: seed {seed}: aggregator not locked yet; cell 9 re-finalises it with the lock (no retraining)')
    cmd = ['python', 'scripts/train_aim2_dl.py', '--config', CONFIG, '--model-seed', str(seed), '--recipe', RECIPE,
           '--train-dir', f'{LOCAL}/train', '--val-dir', f'{LOCAL}/val',
           '--ckpt-dir', f'{CKPT_LOCAL}/{CONFIG}/seed{seed}', '--mirror-dir', run,
           '--out-dir', f'{OUT_LOCAL}/{CONFIG}/seed{seed}', '--aggregator', agg]
    log = open(f'{LOGS}/train_{CONFIG}_seed{seed}.log', 'a')
    return subprocess.Popen(cmd, cwd='/content/Code', stdout=log, stderr=subprocess.STDOUT)

procs = {seed: launch(seed) for seed in SEEDS}
restarts = {seed: 0 for seed in SEEDS}
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    for seed in SEEDS:
        rc = procs[seed].poll()
        if rc not in (None, 0) and restarts[seed] < MAX_RESTARTS:
            restarts[seed] += 1
            print(time.strftime('%H:%M'), seed, f'exited {rc}; restart {restarts[seed]}/{MAX_RESTARTS} (resumes)')
            procs[seed] = launch(seed)
        h = f'{RESULTS}/{CONFIG}/seed{seed}/history.csv'
        tail = open(h).read().strip().splitlines()[-1] if os.path.exists(h) else 'no pass yet'
        print(time.strftime('%H:%M'), seed, tail[:160])
print({s: p.returncode for s, p in procs.items()}, 'restarts', restarts)
if any(p.returncode for p in procs.values()):
    print('WARNING: a seed still failed; read its log in', LOGS, 'and re-run this cell to resume')

In [ ]:
# 9. Summary; lock the aggregator from M seed 42 (pre-declared rule, validation only); re-finalise mismatches
import pandas as pd
for cfg in ('M', 'P4'):
    for seed in SEEDS:
        run = f'{RESULTS}/{cfg}/seed{seed}'
        if os.path.exists(f'{run}/metrics.json'):
            m = json.load(open(f'{run}/metrics.json'))
            print(cfg, seed, 'val AUC', round(m['best_val_auc_mean_agg'], 4), 'best pass', m['best_pass'],
                  'passes', m['passes_run'], m['stop_reason'], 'agg', m['aggregator'], 'thr', m['threshold'])
            commits = set(m.get('git_commits') or []) | {m.get('git_commit')}
            if commits != {COMMIT}:  # protocol: one C_train for every invocation (resumes included) of every run
                print(f'WARNING: {cfg} seed {seed} was trained (or resumed) at {sorted(map(str, commits))}, not only '
                      f'C_train {COMMIT} ({TRAIN_REF}): protocol breach (predict_aim2_dl.py refuses it for test); '
                      'record it in an addendum')
            if not m.get('prereg_freeze_first') or m.get('allow_unfrozen'):
                print(f'WARNING: {cfg} seed {seed} did not start under the frozen pre-registration '
                      '(never reportable; predict_aim2_dl.py refuses it for test)')
        elif os.path.exists(f'{run}/history.csv'):
            print(cfg, seed, 'running:', len(pd.read_csv(f'{run}/history.csv')), 'passes')
# The lock records M seed 42's best checkpoint: predict_aim2_dl.py re-applies the rule to rule.aucs, refuses a
# lock not from M/seed42, and refuses M seed 42 test inference from any other checkpoint. If M seed 42 is ever
# retrained, delete AGGREGATOR_LOCK.json and re-run this cell before any test inference (pre-registration).
m42 = f'{RESULTS}/M/seed42/postproc.json'
if os.path.exists(m42) and not os.path.exists(AGG_LOCK):
    pp = json.load(open(m42))
    json.dump({'aggregator': pp['aggregator'], 'rule': pp['aggregator_rule'], 'from': m42,
               'best_ckpt_sha256': pp['best_ckpt_sha256'], 'postproc_sha256': sha256(m42),
               'train_ref': TRAIN_REF, 'train_commit': COMMIT,
               'locked': time.strftime('%Y-%m-%dT%H:%M:%S')}, open(AGG_LOCK, 'w'), indent=2)
    print('LOCKED aggregator', pp['aggregator'], pp['aggregator_rule']['aucs'])
# Every finished run must use the locked aggregator. A run that finalised with 'auto' before the lock
# existed is re-finalised with the locked choice: the trainer resumes with done=True and only rewrites
# val_predictions / postproc / metrics (aggregator is not part of the config hash).
if os.path.exists(AGG_LOCK):
    LOCKED = json.load(open(AGG_LOCK))['aggregator']
    for cfg in ('M', 'P4'):
        if cfg != CONFIG:
            print(f'{cfg}: switch CONFIG to {cfg} and re-run cells 4-5 + this cell to check it'); continue
        for seed in SEEDS:
            run = f'{RESULTS}/{cfg}/seed{seed}'
            if not os.path.exists(f'{run}/postproc.json'):
                continue
            got = json.load(open(f'{run}/postproc.json'))['aggregator']
            if got == LOCKED:
                continue
            print(f'{cfg} seed {seed}: aggregator {got} != locked {LOCKED}; re-finalising (no retraining)')
            cmd = (f'python scripts/train_aim2_dl.py --config {cfg} --model-seed {seed} --recipe {RECIPE} '
                   f'--train-dir {LOCAL}/train --val-dir {LOCAL}/val --ckpt-dir {CKPT_LOCAL}/{cfg}/seed{seed} '
                   f'--mirror-dir {run} --out-dir {OUT_LOCAL}/{cfg}/seed{seed} --aggregator {LOCKED}')
            !{cmd} 2>&1 | tee -a {LOGS}/train_{cfg}_seed{seed}.log
            got = json.load(open(f'{run}/postproc.json'))['aggregator']
            assert got == LOCKED, f'{cfg} seed {seed} still has aggregator {got}'
    print('all finished runs of', CONFIG, 'use the locked aggregator', LOCKED)

**Next steps (on the Mac, not here)**, in the pre-registered order (no test metric before step 5 is done
for every run):

1. Download each `RESULTS/<cfg>/seed<k>/` (best.pt, last.pt, metrics.json, postproc.json, history.csv,
   val_predictions.parquet) into `Code/models/aim2_dl_olsen_v1/<cfg>/seed<k>/`, and `RESULTS/AGGREGATOR_LOCK.json`
   and `RESULTS/RECIPE_LOCK.json` into `Code/models/aim2_dl_olsen_v1/`.
2. Tag the code freeze and check it out on a clean tree (`predict_aim2_dl.py --split test` refuses otherwise).
3. Pack the test cache with the frozen code (`python scripts/derive_dl_signals.py pack --splits test`); it must
   have the same front-end / stage-B / EDR versions as train and val (predict refuses otherwise).
4. Parity gate, per (config, seed):
   `python scripts/predict_aim2_dl.py --split val --refit-postproc --ckpt models/aim2_dl_olsen_v1/<cfg>/seed<k>/best.pt
   --postproc models/aim2_dl_olsen_v1/<cfg>/seed<k>/postproc.json --data-dir ~/thesis_dl_cache/model_v1/val
   --aggregator-lock models/aim2_dl_olsen_v1/AGGREGATOR_LOCK.json --prereg '<vault note>'`
   (writes `postproc_mps.json` into the run folder). Run it at the freeze tag (test inference refuses a refit made
   by other code). Add `--no-amp` for a config retrained under the NaN fallback (fp32): predict refuses a refit or
   test inference on MPS at another precision than training. If any run prints PARITY FAILED, MPS is abandoned for every run:
   redo this step with `--device cpu --no-amp` (`postproc_cpu.json`) and run step 5 on CPU too. The failure is kept in
   `PARITY_FAILED_mps.json` (run folder and `models/aim2_dl_olsen_v1/`); from then on predict refuses MPS refits and
   MPS test inference for every run, and evaluate refuses MPS predictions. If the CPU fp32 fallback fails too, stop
   and record it in an addendum: the pre-registration defines no further fallback.
5. Test inference, once per (config, seed), into the run folder (no `--out-dir`):
   `python scripts/predict_aim2_dl.py --split test --freeze-tag <tag> --ckpt models/aim2_dl_olsen_v1/<cfg>/seed<k>/best.pt
   --postproc models/aim2_dl_olsen_v1/<cfg>/seed<k>/postproc_mps.json --data-dir ~/thesis_dl_cache/model_v1/test
   --aggregator-lock models/aim2_dl_olsen_v1/AGGREGATOR_LOCK.json --prereg '<vault note>'`
   (writes test_predictions.parquet, test_sec_probs.npy and predict_test.json; no metrics). Same precision as
   step 4 (`--no-amp` for a NaN-fallback config).
6. Only then the matched LightGBM cells on test: `python scripts/fit_aim2_matched_cells.py --cell ecg_only --prereg
   '<vault note>' ...` and `--cell ecg_belt` (each refuses until all three DL seeds of its config have test outputs).
7. `python scripts/evaluate_aim2_dl.py --split test --prereg '<vault note>'` (contrasts, gap and short-hypopnoea
   endpoint are fixed by the pre-registration).